In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table, format_table, aggregate_results
from utils.constants import PRED_COLS

In [3]:
cols = [
    "model",
    "use_instructions",
    "type_of_demonstrations",
    "number_of_demonstrations",
    "total_accuracy",
    "theme_accuracy",
    "theme_precision",
    "theme_recall",
    "theme_f1",
    "topic_accuracy",
    "topic_precision",
    "topic_recall",
    "topic_f1",
    "concept_accuracy",
    "concept_precision", 
    "concept_recall",
    "concept_f1",
]

def score_table(jobs, cols, measure_hallucination_detection=True, mark_gen_errors_as_hallucinations=False):

    print(f"Processing {len(jobs)} jobs.")    
    table = {col_name: [] for col_name in cols}
    num_parse_errors = 0
    num_gen_errors = 0
    num_rows = 0
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")

        num_rows += len(df)

        # LLM failed to produce proper json
        mask_parse_errors = ~(df[PRED_COLS] != '"PARSE ERROR"').all(axis=1)
        num_parse_errors += list(mask_parse_errors).count(True)

        # LLM did not follow instructions, also includes parse errors
        mask_gen_errors = ~df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)
        num_gen_errors += list(mask_gen_errors).count(True) - list(mask_parse_errors).count(True)

        if mark_gen_errors_as_hallucinations:
            # Set labels
            df.loc[mask_gen_errors, ['themeCorrect', 'topicCorrect', 'usesAdditionalConcepts']] = ['"no"', '"no"', '"yes"']
        else: 
            # Remove erroneous rows
            df = df[~mask_gen_errors]

        config = get_config(job["config_json"])

        # Append run configs
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])

        # Append accuracy scores
        for name, score in calculate_accuracy(df).items():
            table[name].append(score)

        # Append precision, recall, f1 per label
        metrics = calculate_metrics(df)
        for name, score in calculate_metrics(df, measure_hallucination_detection=measure_hallucination_detection).items():
            table[name].append(score)

    print(f"Processed {num_rows} rows.")
    print(f"Number of rows with JSON parse errors: {num_parse_errors}.")
    print(f"Number of rows where LLM failed to adhere to response schema: {num_gen_errors}.")
    print(f"Total number of erroneous rows: {num_parse_errors + num_gen_errors}.")
    print(f"Error rate: {(num_parse_errors + num_gen_errors) / num_rows}")

    return table

In [4]:
def bold_extreme_values(s, by_model=True):
    # Bold max for mean

    if 'mean' not in s.name and 'std' not in s.name:
        return ['' for v in s]


    if not by_model:
        if "mean" in s.name:
            is_max = s == s.max()
            return ['font-weight: bold' if v else '' for v in is_max]
        if "std" in s.name:
            is_min = s == s.min()
            return ['font-weight: bold' if v else '' for v in is_min]
    
    font_array = []

    model_level = s.index.names.index('model')
    models = s.index.get_level_values(model_level)
    models = pd.Series(list(models)).unique()

    idx = pd.IndexSlice
    
    for model in models:   
        values_by_model = s.loc[idx[model]]
        
        if "mean" in s.name:
            is_max = values_by_model == values_by_model.max()
            font_array += ['font-weight: bold' if v else '' for v in is_max]
        if "std" in s.name:
            is_min = values_by_model == values_by_model.min()
            font_array += ['font-weight: bold' if v else '' for v in is_min]

    return font_array

In [6]:
# Collect raw data
basepath = "./outputs/v6"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

res = pd.DataFrame(score_table(jobs_list, cols, mark_gen_errors_as_hallucinations=False))
res = prettify_table(res)

Processing 330 jobs.
Processed 175350 rows.
Number of rows with JSON parse errors: 1054.
Number of rows where LLM failed to adhere to response schema: 199.
Total number of erroneous rows: 1253.
Error rate: 0.007145708582834332


In [7]:
agg = aggregate_results(res, ["model", "number_of_demonstrations", "type_of_demonstrations", "use_instructions"], cols[4:])

# Drop count
#agg2 = agg2.loc[idx[use_models, :, :, :], (["theme_f1", "topic_f1", "concept_f1"], ["mean"])]
agg = format_table(agg)

#agg2
# Highlight max values for each column in model group
#agg2.style.apply(bold_extreme_values, axis=0)
#print(agg2.style.apply(bold_extreme_values, axis=0).to_latex())
#print(agg2.to_latex(escape=True, sparsify=True, float_format="%.3f"))

pd.options.display.max_columns = None
pd.options.display.max_rows = None

agg.loc[pd.IndexSlice[['Llama-70B'], :, :, :], (slice(None), ["std"])]

total accuracy  \
                                                                                      std   
model     number of demonstrations type of demonstrations use instructions                  
Llama-70B 0                        none                   yes                    0.000000   
          1                        negative               no                     0.064545   
                                                          yes                    0.052768   
                                   positive               no                     0.081340   
                                                          yes                    0.011620   
          6                        mixed                  no                     0.059698   
                                                          yes                    0.019204   
                                   negative               no                     0.050132   
                                                          yes                    0.016876   
                                   positive               no                     0.058551   
                                                          yes                    0.006997   

                                                                           theme accuracy  \
                                                                                      std   
model     number of demonstrations type of demonstrations use instructions                  
Llama-70B 0                        none                   yes                    0.000000   
          1                        negative               no                     0.012361   
                                                          yes                    0.023201   
                                   positive               no                     0.009272   
                                                          yes                    0.013334   
          6                        mixed                  no                     0.010826   
                                                          yes                    0.006894   
                                   negative               no                     0.013891   
                                                          yes                    0.006411   
                                   positive               no                     0.006943   
                                                          yes                    0.003434   

                                                                           theme precision  \
                                                                                       std   
model     number of demonstrations type of demonstrations use instructions                   
Llama-70B 0                        none                   yes                     0.000000   
          1                        negative               no                      0.013302   
                                                          yes                     0.011606   
                                   positive               no                      0.018069   
                                                          yes                     0.008210   
          6                        mixed                  no                      0.015683   
                                                          yes                     0.010361   
                                   negative               no                      0.023416   
                                                          yes                     0.009027   
                                   positive               no                      0.011216   
                                                          yes                     0.003178   

                                                                           theme recall  \
                                                             

In [11]:
flattened = agg.copy()

flattened.columns = [" ".join(list(cols)) for cols in flattened.columns]
flattened = flattened.reset_index()

#flattened.loc[62]
flattened.to_csv("./data/metrics.csv", sep=";")